In [ ]:
!pip -q install -U transformers accelerate datasets peft trl bitsandbytes sentencepiece rouge_score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 93.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.2 MB/s eta 0:00:00


In [ ]:
!nvidia-smi

Mon Mar 30 20:15:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

torch.cuda.empty_cache()
gc.collect()

model_name = "meta-llama/Llama-3.2-1B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()
model.config.use_cache = True

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [ ]:
import pandas as pd
import torch
from rouge_score import rouge_scorer

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# Inference

In [ ]:
import sys
import csv
from datasets import Dataset

csv.field_size_limit(sys.maxsize)

test_df = pd.read_csv("test_robotics.csv", engine="python").dropna(subset=["x", "summary"])

test_dataset = Dataset.from_pandas(test_df, preserve_index=False)

max_ctx = 2048
max_new = 200
max_input = max_ctx - max_new

prefix = "Summarize the following scientific paper excerpt in 3-5 sentences.\n\n"
suffix = "\n\nSummary:\n"

prefix_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]
suffix_ids = tokenizer(suffix, add_special_tokens=False)["input_ids"]


In [ ]:
def format_example(row):
    x_text = str(row["x"])
    y_text = str(row["summary"]).strip() + tokenizer.eos_token

    y_ids = tokenizer(y_text, add_special_tokens=False)["input_ids"]

    budget_for_x = max_input - len(prefix_ids) - len(suffix_ids) - len(y_ids)

    if budget_for_x < 0:
        y_ids = y_ids[: max(32, max_input // 4)]
        budget_for_x = max_input - len(prefix_ids) - len(suffix_ids) - len(y_ids)

    x_ids = tokenizer(x_text, add_special_tokens=False)["input_ids"][:max(0, budget_for_x)]

    prompt_ids = prefix_ids + x_ids + suffix_ids
    input_ids = prompt_ids + y_ids
    attention_mask = [1] * len(input_ids)

    labels = [-100] * len(prompt_ids) + y_ids

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


In [ ]:
test_dataset = test_dataset.map(
    format_example,
    remove_columns=test_dataset.column_names,
)

Map:   0%|          | 0/180 [00:00<?, ? examples/s]

In [ ]:
!unzip llama32_1b_nlp_lora.zip -d /content/llama32_1b_nlp_lora/

Archive:  llama32_1b_nlp_lora.zip
   creating: /content/llama32_1b_nlp_lora/checkpoint-180/
   creating: /content/llama32_1b_nlp_lora/final_adapter/
  inflating: /content/llama32_1b_nlp_lora/final_adapter/README.md  
  inflating: /content/llama32_1b_nlp_lora/final_adapter/tokenizer_config.json  
  inflating: /content/llama32_1b_nlp_lora/final_adapter/chat_template.jinja  
  inflating: /content/llama32_1b_nlp_lora/final_adapter/adapter_config.json  
  inflating: /content/llama32_1b_nlp_lora/final_adapter/adapter_model.safetensors  
  inflating: /content/llama32_1b_nlp_lora/final_adapter/tokenizer.json  
  inflating: /content/llama32_1b_nlp_lora/checkpoint-180/trainer_state.json  
  inflating: /content/llama32_1b_nlp_lora/checkpoint-180/optimizer.pt  
  inflating: /content/llama32_1b_nlp_lora/checkpoint-180/README.md  
  inflating: /content/llama32_1b_nlp_lora/checkpoint-180/scheduler.pt  
  inflating: /content/llama32_1b_nlp_lora/checkpoint-180/training_args.bin  
  inflating: /content/

In [ ]:
from peft import PeftModel

model_adapter = PeftModel.from_pretrained(model, "/content/llama32_1b_nlp_lora/final_adapter")
model_adapter.eval()

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lor

In [ ]:
from tqdm import tqdm

all_scores = []
rows_output = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df)):
    text = str(row["x"])

    budget_for_text = max_input - len(prefix_ids) - len(suffix_ids)
    text_ids = tokenizer(text, add_special_tokens=False)["input_ids"][:budget_for_text]
    truncated_text = tokenizer.decode(text_ids, skip_special_tokens=True)

    prompt = prefix + truncated_text + suffix
    inputs = tokenizer(prompt, return_tensors="pt", truncation=False).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    pred = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    ref = str(row["summary"]).strip()

    score = scorer.score(ref, pred)["rougeL"]

    rows_output.append({
        "index": i,
        "prediction": pred,
        "reference": ref,
        "rougeL_precision": score.precision,
        "rougeL_recall": score.recall,
        "rougeL_f1": score.fmeasure,
    })

    all_scores.append(score)

    print(f"\n--- Sample {i} ---")
    print(f"F1: {score.fmeasure:.4f} | P: {score.precision:.4f} | R: {score.recall:.4f}")


results_df = pd.DataFrame(rows_output)
results_df.to_csv("detailed_test_results_nlp_with_adapter.csv", index=False)


precision = sum(s.precision for s in all_scores) / len(all_scores)
recall = sum(s.recall for s in all_scores) / len(all_scores)
f1 = sum(s.fmeasure for s in all_scores) / len(all_scores)

print("\n=== FINAL TEST RESULTS ===")
print(f"ROUGE-L Precision: {precision:.4f}")
print(f"ROUGE-L Recall:    {recall:.4f}")
print(f"ROUGE-L F1:        {f1:.4f}")

  1%|          | 1/180 [00:09<29:02,  9.73s/it]


--- Sample 0 ---
F1: 0.2316 | P: 0.2867 | R: 0.1943


  1%|          | 2/180 [00:18<27:06,  9.14s/it]


--- Sample 1 ---
F1: 0.1775 | P: 0.2113 | R: 0.1531


  2%|▏         | 3/180 [00:25<24:34,  8.33s/it]

...


--- Sample 177 ---
F1: 0.1514 | P: 0.2927 | R: 0.1021


 99%|█████████▉| 179/180 [23:48<00:07,  7.10s/it]


--- Sample 178 ---
F1: 0.2449 | P: 0.4000 | R: 0.1765


100%|██████████| 180/180 [23:56<00:00,  7.98s/it]


--- Sample 179 ---
F1: 0.1992 | P: 0.2294 | R: 0.1761

=== FINAL TEST RESULTS ===
ROUGE-L Precision: 0.2846
ROUGE-L Recall:    0.1899
ROUGE-L F1:        0.2210
